# Minimal repro: Unsloth DPO + Gemma 4 vision — RESOLVED Apr 27, 2026

**Status: Resolved.** End-to-end vision DPO training now works. See "Resolution" section at the bottom for the working stack.

**Original symptom (Apr 26):** `DPOTrainer.train()` hung silently at `Tokenizing 0/52 [00:00<?, ? examples/s]` with both `dataset_num_proc=1` and `dataset_num_proc=2`. CPU at ~0%, GPU at 0%, no error, no progress.

**Setup:** Gemma 4 E4B 4-bit, vision layers unfrozen, LoRA r=16, fresh SFT-warmstart adapter (or fresh LoRA — either reproduces). Synthetic preference pairs: 1024x1024 RGB images + short prompts/chosen/rejected.

**Diagnostics that worked (function-level):**
- `processor(images=..., text=...)` direct call: returns in 0.1s
- `_UnslothDPOTrainer.process_row(features=pairs[0], ...)` direct call: returns in 0s
- Manual Python loop calling `process_row` over all pairs: returns in ~2s

**What hung originally:** `dataset.map(process_row)` inside `DPOTrainer._prepare_dataset` — the same function works fine outside `dataset.map`.

**Three bugs surfaced and resolved over 2 days** via Unsloth issue [#5196](https://github.com/unslothai/unsloth/issues/5196) and contributor @datta0:

1. **Tokenization hang** (this notebook's original symptom) — fixed by `datta0/unsloth@dpo_mp_hang` branch
2. **Data collator schema mismatch** (`ValueError: ...input_ids`) — fixed in the same branch's second push  
3. **AttributeError in vision tower forward** (`'bool' object has no attribute 'all'`) — was specific to `trl 0.22.2`; resolved by upgrading to `trl 0.24.0`

**Environment:** Modal A100 80GB, Linux, Python 3.12.

## 1. Install

In [1]:
%%capture
# Order matters: install everything else FIRST, then force torch+torchvision LAST
# so nothing downstream pulls in a different torch version
%uv pip install pillow==11.3.0 --system
%uv pip install --upgrade transformers --system
%uv pip install --upgrade trl --system
%uv pip install "peft==0.19.1" --system
%uv pip install --upgrade datasets --system
%uv pip install unsloth-zoo --system

# Apply Datta's fix branch (--no-deps means won't touch torch)
%uv pip uninstall unsloth --system
%uv pip install "git+https://github.com/datta0/unsloth@dpo_mp_hang" --no-deps --system

# NOW force torch+torchvision matching pair from cu129 index, last
%uv pip install --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu129 --system

!rm -rf /root/unsloth_compiled_cache

In [2]:
import subprocess
result = subprocess.run(['pip', 'show', 'torch', 'torchvision'], capture_output=True, text=True)
print(result.stdout)

Name: torch
Version: 2.11.0+cu129
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvshmem-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, peft, torchaudio, torchvision, unsloth_zoo
---
Name: torchvision
Version: 0.26.0+cu129
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: soumith@pytorch.org
License: BSD
Location: /usr/local/lib/python3.12/site-packages
Requires: numpy, pillow, torch
Required-by: 



In [3]:
%uv pip install bitsandbytes --system

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 13ms
Note: you may need to restart the kernel to use updated packages.


In [4]:
import unsloth
import importlib.metadata as md
for pkg in ["bitsandbytes", "unsloth", "unsloth_zoo", "trl", "transformers", "datasets", "peft", "torch", "pillow"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

print(f"\nunsloth location: {unsloth.__file__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
bitsandbytes: 0.49.2
unsloth: 2026.4.8
unsloth_zoo: 2026.4.9
trl: 0.24.0
transformers: 5.5.0
datasets: 4.3.0
peft: 0.19.1
torch: 2.11.0+cu129
pillow: 12.1.1

unsloth location: /usr/local/lib/python3.12/site-packages/unsloth/__init__.py


## 2. Imports

In [5]:
from unsloth import FastVisionModel       # MUST be first
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from PIL import Image
import torch
import time
import os

## 3. Load Gemma 4 E4B 4-bit + fresh LoRA adapter

Use Unsloth's `FastVisionModel.get_peft_model` (NOT `PeftModel.from_pretrained`, which fails on `Gemma4ClippableLinear` module resolution).

In [6]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-e4b-it",      
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print("Trainable params:")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base loaded. GPU: 10.9 GB
Trainable params:
trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129


## 4. Synthetic preference pairs (52 examples)

Just blank 1024x1024 RGB PIL images + short text. Real images and real preferences are not needed to reproduce — the hang is structural, not data-dependent.

Schema: flat `{images: [PIL], prompt: str, chosen: str, rejected: str}` — this is what Unsloth's vision DPO `process_row` expects (it does NOT want chat-template content blocks).

In [7]:
def make_pair(idx: int):
    img = Image.new("RGB", (1024, 1024), color=(idx % 256, 128, 200))
    return {
        "images": [img],
        "prompt": "Describe the image in one sentence.",
        "chosen": f"Image {idx} shows a solid color block.",
        "rejected": f"Image {idx} contains nothing useful at all.",
    }

pairs = [make_pair(i) for i in range(52)]
print(f"Built {len(pairs)} synthetic pairs")
print(f"Sample image: {pairs[0]['images'][0].size}")

Built 52 synthetic pairs
Sample image: (1024, 1024)


## 5. Diagnostic: confirm `process_row` works on its own

This call IS the function that `dataset.map` will invoke 52 times during DPOTrainer prep. It works fine in isolation.

In [8]:
from unsloth_compiled_cache.UnslothDPOTrainer import _UnslothDPOTrainer

start = time.time()
out = _UnslothDPOTrainer.process_row(
    features=pairs[0],
    processing_class=tokenizer,
    max_prompt_length=1024,
    max_completion_length=None,
    add_special_tokens=False,
)
print(f"Single call OK in {time.time()-start:.2f}s. Keys: {list(out.keys())}")

Single call OK in 0.05s. Keys: ['prompt_input_ids', 'pixel_values', 'chosen_input_ids', 'rejected_input_ids']


In [9]:
# Manual loop: call process_row on every pair. Should finish in ~2s.
start = time.time()
processed = []
for p in pairs:
    out = _UnslothDPOTrainer.process_row(
        features=p,
        processing_class=tokenizer,
        max_prompt_length=1024,
        max_completion_length=None,
        add_special_tokens=False,
    )
    processed.append(out)
print(f"Manual loop over {len(pairs)} pairs: {time.time()-start:.2f}s")

Manual loop over 52 pairs: 0.98s


## 6. Repro: DPOTrainer hangs at tokenization step

Wraps `pairs` in a `Dataset` and instantiates DPOTrainer. The trainer's `_prepare_dataset` calls `dataset.map(process_row, ...)` internally — that's where it hangs.

Expected behaviour: tokenization completes in seconds (since `process_row` is fast).

Actual behaviour: progress bar sits at `0/52 [00:00<?, ? examples/s]`, CPU drops to ~0%, GPU at 0%, no progress, no error. Have to interrupt the kernel.

In [10]:
pairs_ds = Dataset.from_list(pairs)
print(f"Dataset: {len(pairs_ds)} examples, columns: {pairs_ds.column_names}")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=pairs_ds,
    args=DPOConfig(
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-6,
        output_dir="/tmp/dpo-repro",
        max_length=2048,
        max_prompt_length=1024,
        remove_unused_columns=False,
        logging_steps=1,
        dataset_num_proc=1,    # also tried 2 — both hang
    ),
)

# fails here ↓
dpo_trainer.train()

Dataset: 52 examples, columns: ['images', 'prompt', 'chosen', 'rejected']


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.693149,0.000000,0.000003,0.000000,-0.000003,-149.356903,-192.535690,-8.792302,-7.490411
2,0.693147,0.000000,0.000000,0.000000,0.000000,-155.021912,-197.751236,-8.354200,-6.880948
3,0.704540,0.190698,0.204769,0.500000,-0.014071,-146.976791,-187.719452,-8.733580,-7.308352
4,0.655879,-0.009127,-0.088159,0.750000,0.079032,-155.379913,-194.939575,-8.665869,-7.205754
5,0.745576,-0.129678,-0.034777,0.500000,-0.094901,-152.506531,-193.828094,-8.936730,-7.337120
6,0.685129,0.018552,-0.017125,0.500000,0.035677,-151.896240,-192.751526,-8.986294,-7.349975
7,0.538729,-0.139367,-0.492425,1.000000,0.353058,-149.490005,-192.556274,-9.071600,-7.343634
8,0.660499,-0.080231,-0.159048,0.500000,0.078817,-146.956848,-187.464874,-8.865355,-7.357415
9,0.551894,0.028682,-0.290693,1.000000,0.319375,-150.358627,-193.251312,-8.851860,-7.298950
10,0.428196,0.262462,-0.397896,1.000000,0.660358,-150.427643,-198.460434,-8.685949,-7.138791


Unsloth: Restored added_tokens_decoder metadata in /tmp/dpo-repro/checkpoint-39/tokenizer_config.json.


TrainOutput(global_step=39, training_loss=0.20546490636218387, metrics={'train_runtime': 777.86, 'train_samples_per_second': 0.201, 'train_steps_per_second': 0.05, 'total_flos': 0.0, 'train_loss': 0.20546490636218387, 'epoch': 3.0})

## Resolution — working stack (Apr 27, 2026)

After resolving 3 cascading bugs, this stack runs vision DPO end-to-end on Gemma 4 E4B:

- `unsloth`: `git+https://github.com/datta0/unsloth@e96d05bad233052a30f894c5050eb7ec2e65ebc5` (commit-pinned; awaiting merge to mainline)
- `trl`: 0.24.0 (NOT 0.22.2 from the original report — bug 3 lived there)
- `torch`: 2.11.0+cu129 + `torchvision`: 0.26.0+cu129 (matching CUDA builds, install with `--index-url https://download.pytorch.org/whl/cu129`)
- `transformers`: 5.5.0
- `datasets`: 4.3.0
- `peft`: 0.19.1
- `unsloth-zoo`: 2026.4.9
- `bitsandbytes`: 0.49.2

**Install order matters** — separate transactions so dependency resolution doesn't downgrade things. See cell 2 above for the working sequence.

**Synthetic test result:** 39-step DPO run on Gemma 4 E4B 4-bit, vision-unfrozen LoRA r=16:

- Loss: 0.6931 (random baseline) → 0.0005 (converged)
- `rewards/accuracies`: 0.0 → 1.0
- `rewards/margin`: 0 → ~7.5

The model overfits to the synthetic preferences as expected — confirms the training pipeline is working end-to-end.

## Original "What I've tried" (kept for posterity)

These were attempts during initial debugging on Apr 26 BEFORE the upstream fixes landed:

1. `dataset_num_proc=1` and `=2` — both hung. Single-process meant no fork issue, but still wedged.
2. Forced `HF_DATASETS_CACHE` to a known-writable persistent path — no change.
3. Explicit `Features({"images": [HFImage()], ...})` schema — Dataset constructed in 0s, trainer still hung at the same step.
4. Pre-tokenize via manual loop, then pass tokenized dataset — TRL's `_prepare_dataset` still called `dataset.map(..., remove_columns=["chosen", "rejected"])` and errored because those columns were gone.

None of these were the right path. The actual fix was upstream library code (Datta's branch) plus a TRL version bump.

## Credits

@datta0 for the multiple fast turnarounds on the unsloth fixes. Issue thread: [#5196](https://github.com/unslothai/unsloth/issues/5196).